## Завдання 1. Математична постановка власного варіанта

Варіант 7 — **Івано-Франківськ**.

Нехай:

* `x1` — кількість виробів А, які виробляються за тиждень;
* `x2` — кількість виробів Б, які виробляються за тиждень.

Необхідно максимізувати тижневий прибуток:

`max Z = 90*x1 + 60*x2`

за таких обмежень.

**Сировина:**

`6*x1 + 4*x2 <= 360`

**Робочий час обладнання:**

`3*x1 + 5*x2 <= 270`

**Електроенергія:**

`1*x1 + 2*x2 <= 125`

**Умова невід'ємності:**

`x1 >= 0`

`x2 >= 0`

Отже, задача полягає у визначенні такої кількості виробів А і Б, яка забезпечує максимальний тижневий прибуток при дотриманні всіх трьох ресурсних обмежень.


In [1]:
import numpy as np
from scipy.optimize import linprog

In [2]:
c = [-90, -60]

A_ub = [
    [6, 4],
    [3, 5],
    [1, 2]
]

b_ub = [
    360,
    270,
    125
]

bounds = [
    (0, None),
    (0, None)
]

res = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs"
)

print("res.x =", res.x)
print("res.fun =", res.fun)
print("Максимальний прибуток =", -res.fun)
print("res.status =", res.status)
print("res.message =", res.message)

res.x = [40. 30.]
res.fun = -5400.0
Максимальний прибуток = 5400.0
res.status = 0
res.message = Optimization terminated successfully. (HiGHS Status 7: Optimal)


## Результат Завдання 2

Функція `linprog` за замовчуванням виконує мінімізацію, тому для задачі максимізації прибутку коефіцієнти цільової функції були взяті зі знаком мінус: `[-90, -60]`.

Отримано:

`res.x = [40, 30]`

`res.fun = -5400`

Справжній максимальний прибуток становить:

`-res.fun = 5400 грн/тиждень`.

Значення `res.status = 0`, тому задача успішно розв'язана, а отриманий результат є оптимальним.


In [3]:
x1, x2 = res.x

profit = 90 * x1 + 60 * x2

print(f"Виріб A: {x1:.2f} од./тиждень")
print(f"Виріб B: {x2:.2f} од./тиждень")
print(f"Максимальний прибуток: {profit:.2f} грн/тиждень")

Виріб A: 40.00 од./тиждень
Виріб B: 30.00 од./тиждень
Максимальний прибуток: 5400.00 грн/тиждень


## Завдання 3. Інтерпретація оптимального розв'язку

Оптимальний план виробництва становить **40 одиниць виробу А** та **30 одиниць виробу Б** на тиждень.

Тижневий прибуток становить:

`90 · 40 + 60 · 30 = 5400 грн`.

Отже, за заданих ресурсних обмежень максимальний прибуток виробничої дільниці становить **5400 грн на тиждень**.

У цьому випадку оптимальні значення є цілими, тому додаткове округлення не потрібне. Якби `linprog` повернув дробові значення, просте округлення не гарантувало б оптимального цілочислового плану, оскільки після округлення можна порушити ресурсні обмеження або отримати план із меншим прибутком. Для гарантовано найкращого цілочислового рішення потрібен метод цілочислової оптимізації.


In [4]:
print("Використання ресурсів:")

raw_material = 6 * x1 + 4 * x2
work_time = 3 * x1 + 5 * x2
electricity = x1 + 2 * x2

print(f"Сировина: {raw_material:.2f} / 360 кг")
print(f"Робочий час: {work_time:.2f} / 270 год")
print(f"Електроенергія: {electricity:.2f} / 125 кВт·год")

print()
print("Slack:", res.slack)

Використання ресурсів:
Сировина: 360.00 / 360 кг
Робочий час: 270.00 / 270 год
Електроенергія: 100.00 / 125 кВт·год

Slack: [ 0.  0. 25.]


## Завдання 4. Активні обмеження

Для оптимального плану `x1 = 40` та `x2 = 30` використання ресурсів становить:

* сировина: `360 кг` із `360 кг`;
* робочий час: `270 год` із `270 год`;
* електроенергія: `100 кВт·год` із `125 кВт·год`.

Відповідно:

`res.slack = [0, 0, 25]`.

Активними є два обмеження: **сировина та робочий час**, оскільки вони використовуються повністю. Вони є вузькими місцями виробництва.

Неактивним є обмеження на **електроенергію**. Запас становить `25 кВт·год`, тому збільшення доступної електроенергії саме по собі не повинно збільшити прибуток за поточного плану.

Для розширення виробництва доцільно насамперед розглядати збільшення активних ресурсів — сировини або робочого часу. Саме вони обмежують подальше збільшення прибутку.


In [5]:
b_15 = [414, 270, 125]

res_active = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_15,
    bounds=bounds,
    method="highs"
)

print("Новий план:", res_active.x)
print("Нове значення res.fun:", res_active.fun)
print("Новий максимальний прибуток:", -res_active.fun)
print("Новий slack:", res_active.slack)

Новий план: [55. 21.]
Нове значення res.fun: -6210.0
Новий максимальний прибуток: 6210.0
Новий slack: [ 0.  0. 28.]


## Завдання 5.1. Зміна активного ресурсу

Ліміт сировини було збільшено на 15%:

`360 · 1.15 = 414 кг`.

Після повторного розв'язання отримано новий оптимальний план:

* виріб А — **55 одиниць**;
* виріб Б — **21 одиниця**;
* максимальний прибуток — **6210 грн/тиждень**.

Порівняно з початковим планом `40 А + 30 Б` прибуток збільшився з **5400 грн до 6210 грн**, тобто на **810 грн**.

Також змінилася структура випуску: частка виробу А збільшилася, а кількість виробу Б зменшилася. Це пояснюється тим, що виріб А приносить більший прибуток на одиницю та після збільшення запасу сировини стає вигідніше використовувати додаткову сировину для його виробництва.

Результат підтверджує, що активне ресурсне обмеження є вузьким місцем: його розширення дозволило збільшити максимальний прибуток.


In [6]:
b_50 = [360, 270, 187.5]

res_inactive = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_50,
    bounds=bounds,
    method="highs"
)

print("Новий план:", res_inactive.x)
print("Новий максимальний прибуток:", -res_inactive.fun)
print("Новий slack:", res_inactive.slack)

Новий план: [40. 30.]
Новий максимальний прибуток: 5400.0
Новий slack: [ 0.   0.  87.5]


## Завдання 5.2. Зміна неактивного ресурсу

Ліміт електроенергії було збільшено на 50%:

`125 · 1.5 = 187.5 кВт·год`.

Після повторного розв'язання отримано:

* виріб А — **40 одиниць**;
* виріб Б — **30 одиниць**;
* максимальний прибуток — **5400 грн/тиждень**.

Отже, оптимальний план та прибуток не змінилися.

Причина полягає в тому, що електроенергія була **неактивним обмеженням**. У початковому оптимальному плані використовується лише `100 кВт·год` із доступних `125 кВт·год`, тому вже існує запас `25 кВт·год`.

Збільшення ресурсу, який не є вузьким місцем, не створює додаткових можливостей для збільшення виробництва. Основними обмеженнями залишаються сировина та робочий час. Це підтверджує ідею з Лекції 19: **активне обмеження є ресурсом, який справді обмежує виробництво та заслуговує першочергової уваги при розширенні**.


## Контрольні питання

**1. Чому `scipy.optimize.linprog` мінімізує цільову функцію і що потрібно зробити для максимізації?**

`linprog` розв'язує задачу мінімізації. Якщо необхідно максимізувати прибуток `90*x1 + 60*x2`, коефіцієнти цільової функції потрібно помножити на `-1` та передати як `[-90, -60]`. Після отримання результату справжній максимальний прибуток визначається як `-res.fun`.

**2. Чому обмеження `>=` потрібно домножити на -1?**

`linprog` приймає ресурсні обмеження у формі `A_ub · x <= b_ub`. Тому нерівність виду `a*x >= b` потрібно помножити на `-1`. Наприклад, `2*x >= 10` перетворюється на `-2*x <= -10`.

**3. Що таке активне та неактивне обмеження?**

Активне обмеження виконується точно на межі, тому його `slack` дорівнює нулю. Воно є вузьким місцем виробництва. Неактивне обмеження має додатний запас, тому збільшення такого ресурсу саме по собі не покращує оптимальний результат.

**4. Що показав експеримент із Завдання 5?**

Збільшення активного ресурсу — сировини — на 15% дозволило збільшити прибуток із 5400 грн до 6210 грн на тиждень. Натомість збільшення неактивного ресурсу — електроенергії — на 50% не змінило ні оптимальний план, ні прибуток. Отже, результати узгоджуються з тим, що саме активні обмеження є вузькими місцями виробництва.


## Висновок

У практичній роботі було сформульовано та розв'язано задачу лінійного програмування для виробничої дільниці в Івано-Франківську, варіант 7.

Було визначено дві змінні: кількість виробів А та Б. Цільовою функцією є максимізація тижневого прибутку за умов обмеженої кількості сировини, робочого часу та електроенергії.

За допомогою `scipy.optimize.linprog` з методом `highs` отримано оптимальний план виробництва: **40 одиниць виробу А та 30 одиниць виробу Б**. Максимальний тижневий прибуток становить **5400 грн**.

Аналіз `res.slack` показав, що сировина та робочий час є активними обмеженнями та виступають вузькими місцями виробництва. Електроенергія є неактивним ресурсом і має запас `25 кВт·год`.

У what-if аналізі збільшення запасу сировини на 15% призвело до зміни оптимального плану до **55 одиниць виробу А та 21 одиниці виробу Б** і збільшення прибутку до **6210 грн/тиждень**. Натомість збільшення запасу електроенергії на 50% не змінило оптимального плану та прибутку.

Таким чином, проведений аналіз підтвердив практичне значення активних обмежень: саме ресурси, використані повністю, є першочерговими кандидатами для розширення виробництва.
